# 05 — Parquet sharding and LMDB
Challenge: tokenized histories are variable-length and cannot all be loaded for every training batch. Upstream writes one user per Parquet row, grouped by event-count band, and stores user-to-(band, shard, row) mappings plus token counts in LMDB. This is existing functionality, not our contribution. Run after runner setup.

In [ ]:
from pathlib import Path
import json
roots=[Path('runs/research_decomposed'),Path('runs/research_baseline'),Path('runs/research_tokenizer')]
root=next(p for p in roots if (p/'tok/shard_manifest.json').exists())
shard=root/'tok'
print((shard/'shard_manifest.json').read_text())

## Inspect a shard
Arrow large-list arrays store per-user variable-length arrays; event_offsets delimit events within each user's flat arrays. Parquet compression reduces disk footprint but a random read may decode an entire shard in this implementation.

In [ ]:
import pyarrow.parquet as pq
p=next((shard/'shards').glob('*.parquet'))
f=pq.ParquetFile(p)
print(p.name,f.metadata.num_rows,f.schema_arrow)

## LMDB lookup and lazy dataset
The index provides metadata without materializing token arrays. The dataset uses an LRU cache for decoded Arrow shards. get_many groups rows by shard and restores requested order.

In [ ]:
from pragmatiq.data.sharding import UserIndex
from pragmatiq.data.dataset import ShardDataset
idx=UserIndex(shard)
uid=idx.order[0]
m=idx.meta(uid)
print({k:v for k,v in vars(m).items() if k!='user_id'})
ds=ShardDataset(shard,cache_bytes=256*1024**2)
r=ds.get(uid)
print(r.key_ids.shape,r.event_offsets.shape,r.n_events,r.n_tokens)
ds.close(); idx.close()

## Run validation and illustrative timings
Compare all manifest counts with LMDB; verify array equality for individual versus grouped reads. A single timing pass with uncontrolled OS cache is NOT a reliable speedup benchmark.

In [ ]:
import subprocess,sys
subprocess.run([sys.executable,'research/experiments/05_storage_analysis.py','--root',str(root)],check=True)

In [ ]:
import pandas as pd
r=json.loads(Path('research/results/05_storage_metrics.json').read_text())
display(pd.DataFrame(r['bands']))
display(pd.DataFrame(r['shards']))
print(r['sample'])

## Future research (outside six hours)
Investigate byte-aware shard sizing, distributed customer-hash partitions, global vocabulary coordination, memory-mapped layouts and bounded caches. Benchmark cold/warm reads separately with repeated trials before claiming gains.